In [1]:
import sys
import math
import numpy as np # v1.19.5
import tensorflow as tf # version 2.2.0
from tensorflow import keras


from io import UnsupportedOperation
import pickle as pkl


# Add shared location for auxillary functions
sys.path.insert(1, './../AuxillaryFunctions')


#import matplotlib.pyplot as plt

from GenerateClassDataFuncs import GenerateEvalData_OREBA
from GenerateClassDataFuncs import GenerateEvalData_ClemCafe


from EvaluationMethodFuncs import DongEval_PtVsPt
from EvaluationMethodFuncs import KyritEval_PtVsWindow
from EvaluationMethodFuncs import TimePoint2Window
from EvaluationMethodFuncs import Duration_WindowVsWindow
from EvaluationMethodFuncs import PrintStats


In [2]:
DEBUG_PRINTS = 0
DATA_FREQ=64 # Heyd Model Always at 64 Hz


if True:
	MODEL_FILE = 'Models/HeydCNN_CC_0/models/fold1'
	
	
	CUT_sec = 2 # window is always 128 (2 x 64 hz)
	# STRIDE_sec = float(4/64) # move every point at 64 hz
	# CURRENTLY IGNORING STRIDE
	if DEBUG_PRINTS==1:
		print("!!! CURRENTLY HARD-CODING STRIDE INPUT !!!")
	# STRIDE_sec = float(sys.argv[5]) # move every point at 64 hz
	STRIDE_sec = float(8.0/64.0) # move every point at 64 hz
	STRIDE = STRIDE_sec * DATA_FREQ
	FOLD_INDEX = 1 # Which index to leave out for validation
	FOLDS_TOTAL = 5 # total number of folds to split the data into

	RR_FLAG = 1 # 1 if using striped fold division, 0 if using block fold division


	DATABASE_FLAG = 1 # Must be 1 (Dom Hand Only) or 2 (Both Hands)

	WIN_TOLERANCE = 12
	# default to 0.0 for strict method, otherwise provide [sec] preds can be away from window
	# # DOWN SAMPLE DATA INPUTS
	# DOWN_SAMPLE_FLAG = int(sys.argv[7]) # 1 = Down Sample data, 0 = use raw data
	# DOWN_SAMPLE_RATE = int(sys.argv[8]) # Num of points to consolidate into a single point
	# DOWN_SAMPLE_OFFSET = int(sys.argv[9]) # Offset in DS data; in range [0, DS_Rate) 



	DOWN_SAMPLE_FLAG = 0 # 1 = Down Sample data, 0 = use raw data
	DOWN_SAMPLE_RATE = 1 # Num of points to consolidate into a single point
	DOWN_SAMPLE_OFFSET = 0 # Offset in DS data; in range [0, DS_Rate) 

	# NUMBER OF HANDS
	if DATABASE_FLAG == 1:
		DATABASE_FILEPATH = './../Pickle_Databases/OREBA.pkl'
		RESAMPLE_FLAG_FREQ = 0 # zero if keeping original data frequency
	elif DATABASE_FLAG == 2:
		DATABASE_FILEPATH = './../Pickle_Databases/OREBA_TwoHands.pkl'
		RESAMPLE_FLAG_FREQ = 0 # zero if keeping original data frequency
	elif DATABASE_FLAG == 3:
		DATABASE_FILEPATH = "./../Pickle_Databases/ClemCafe.pkl"
		RESAMPLE_FLAG_FREQ = 64 # new freq since ClemCafe = 15Hz
	else:
		print("!!! WARNING: INVALID DATABASE SELECTION FLAG VALUE OF {} !!!".format(DATABASE_FLAG))
		print("Expects value of:")
		print("\t1 = OneHand OREBA")
		print("\t2 = TwoHand OREBA")
		print("\t3 = ClemCafe Data")
		print("Defaulting to use of ClemCafeData...")
		DATABASE_FILEPATH = "./../Pickle_Databases/ClemCafe.pkl"
	#end of Database switch statement

else:
	CUT = 128 # 2 seconds @ 64 Hz # Window Size to cut out of full meal's data
	STRIDE = 8 # 1/8th of a second step @ 64 Hz
	DATABASE_FILEPATH = './../Pickle_Databases/OREBA.pkl'
	FOLD_INDEX = int(sys.argv[2]) # Which index to leave out for validation
	FOLDS_TOTAL = int(sys.argv[3]) # total number of folds to split the data into




# OLD VERSION OF BITE DETECTOR
# EVAL_METHOD_FLAG = 3
# 	# Used to determine how Local Maximums are determined
# 	# 1 = Strict Kyritsis Method; Must have detection fall within bite window
# 	# 2 = Tolerance Kyritsis Method; Allows detections to be up 
# 	#       to WIN_TOLERANCE seconds away from window
# 	# 3 = Dong et al. Method; Greedy Matches detection to nearest GT point (midpoint of window)
# 	# 4 = Kyritsis Method; But Delays detections by WIN_TOLERANCE seconds later
EVAL_METHOD_FLAG = 2
	# 1 = Dong Eval (Pt vs Pt)
	# 2 = Kyritsis (Pt vs Window)
	# 3 = Duration (Window vs Window)
if EVAL_METHOD_FLAG == 2 or EVAL_METHOD_FLAG == 3:
	GT_WINDOW_FLAG = 1 #mark that we want Windows from GT if using a window metric
else:
	GT_WINDOW_FLAG = 0 # mark that we want time points for GT metrics
# end of Window vs Point GT Flag


# # # # # # KNOBS TO TURN FOR EVAL METHOD # # # # # #
DETECTION_METHOD_FLAG = 1 
	# Used to determine how Local Maximums are determined
	# 1 = Immediate Local Maximum; just considers points around it;
	#       Only triggers once every bite_length_sec
	# 2 = Searches each non-floored segment to find the 
	#       maximum and places the detection there; see segment for additional knobs



# # # # # Use model Predictions to place bite in Window # # # # #
# WIN_TOL now defined as user input [7] 
# WIN_TOLERANCE = 1.00 # seconds leeway to give matching windows
BiteDetectThresh=0.65 # Value used by Kyritsis in Paper to trigger a detection
bite_length_sec = 2 # duration in sec to mark a triggered detection as 
					# a bite and ignore any value in that window
delayed_offset_sec = 0.0
# add a small offset to center the predictions in window

if DEBUG_PRINTS == 1:
	EVAL_METHOD_NAMES = ["Names for Eval methods (valid flags start at 1)",
			 "Dong_PtVsPt",
			 "Kyritsis_PtVsWind",
			 "Duration_WindVsWind"]

	print('CUT_sec Input is ',CUT_sec)
	print('STRIDE_sec input is ',STRIDE_sec)
	print('Training Data coming from file ',DATABASE_FILEPATH)
	print('RR_Flag Input is ',RR_FLAG)
	print('Fold_Index Input is ',FOLD_INDEX)
	print('Folds_Total Input is ',FOLDS_TOTAL)
	print('Resample Rate Input is ',RESAMPLE_FLAG_FREQ)
	print('Eval Method is ',EVAL_METHOD_NAMES[EVAL_METHOD_FLAG])
	print('WIN_TOLERANCE (if needed) is ',WIN_TOLERANCE)
	print('Offset_sec (if needed) is ',delayed_offset_sec)
	print("python setup complete")
# end of DEBUG_PRINTS


# # # # # Read in file raw data # # # # #
if DATABASE_FLAG == 1 or DATABASE_FLAG == 2:
	compiled_eval_data = GenerateEvalData_OREBA(
			int(round(CUT_sec*DATA_FREQ)), int(round(STRIDE_sec*DATA_FREQ)),
			DATABASE_FILEPATH,
			FOLD_INDEX,	FOLDS_TOTAL, FOLD_SPLIT=RR_FLAG,
			RESAMPLE_FLAG=RESAMPLE_FLAG_FREQ,
			GT_WINDOW_FLAG = GT_WINDOW_FLAG
			)
	# NOTE: Pass GT_WINDOW_FLAG into OREBA because the original GT is given in Windows
elif DATABASE_FLAG == 3:
	compiled_eval_data = GenerateEvalData_ClemCafe(
			int(round(CUT_sec*DATA_FREQ)), int(round(STRIDE_sec*DATA_FREQ)),
			DATABASE_FILEPATH,
			FOLD_INDEX, FOLDS_TOTAL, FOLD_SPLIT=RR_FLAG,
			RESAMPLE_FLAG=RESAMPLE_FLAG_FREQ # Frequency of Data Collection in [Hz]
			#, RESAMPLE_FLAG=0, SMOOTHING=0 # optional flags not currently used for Paper Experiment
			)
	# NOTE: Pass GT_WINDOW_FLAG into OREBA because the original GT is given in Windows
	if GT_WINDOW_FLAG == 1:
		for meal_num in range(len(compiled_eval_data)):
			compiled_eval_data[meal_num][3] = TimePoint2Window(np.array(compiled_eval_data[meal_num][3])) # default is 2.5 seconds per bite, centered
	# end of if Converting GT to Windows
#end of Database switch statement

# # # # # Load Network Model # # # # #
model = tf.keras.models.load_model(MODEL_FILE)

print("Finished Initialization and Setup.")

# # # # # Construct Bite Predictions for Bite Indices # # # # #
if DEBUG_PRINTS == 1:
	print("Testing...")

Finished Initialization and Setup.


In [4]:


# Lists for the metrics of each meal evaluated
All_TP=[] # List (one entry per file) of bites that trigger within ground truth
All_FP=[] # List of Bites that trigger but have already been mapped to a TP
All_FN=[] # List of GT windows without a bite detected

# If using Point vs Window Eval, use these two lists
All_FP1=[] # List of Bites that trigger inside of another TP
All_FP2=[] # List of Bites that trigger outside of any GT


for curr_meal in range(len(compiled_eval_data)):

	# Grab current meal data
	[meal_id,features_input,timesteps,currGT_times]=compiled_eval_data[curr_meal] 
	# Get prediction probabilities
	predictions = model.predict(features_input)
	if DEBUG_PRINTS==1:
		predictions.tofile('Predictions_Raw'+str(curr_meal)+'_subj'+str(FOLD_INDEX)+'.csv', sep = ',')
		np.array(timesteps).tofile('timesteps'+str(curr_meal)+'_subj'+str(FOLD_INDEX)+'.csv', sep = ',')
	# End if DEBUG_PRINT
	for i in range(len(predictions)):
		if predictions[i]<BiteDetectThresh:
			predictions[i]=0
	# predictions[predictions<BiteDetectThresh]=0 # floor any predictions below threshold


	if DETECTION_METHOD_FLAG == 1: # First Method: Floor predictions, then find any local maximum (based on its two neighbors), 
		#then detect the largest LM that ensures all LM are either detections or covered up in 2 sec window 

		# Filter out so only local maxima remain
		predictions_LM=[]
		#take care of front edge cases
		if predictions[0]>predictions[1]:
			predictions_LM.append(1)
		else:
			predictions_LM.append(0)

		#iterate over the rest of the data
		for i in range(1,len(predictions)-1):
			#Bias data towards the left most point in the event of equal values
			if predictions[i]>predictions[i-1] and predictions[i]>=predictions[i+1]:
				predictions_LM.append(1)
			else:
				predictions_LM.append(0)
		#take care of back edge cases
		if predictions[-1]!=0 and predictions[-1]>=predictions[-2]:
			predictions_LM.append(1)
		else:
			predictions_LM.append(0)

		predictions_LM=np.array(predictions_LM)


		Detections=[] # initialize as empty list, add detections as they are found
		for i in range(len(predictions_LM)):
			# if it is a local maximum that needs to be either covered or detected
			if predictions_LM[i]==1:
				# find maximum value within duration ahead to see if it will be covered
				running_max_index=i
				#check the duration ahead for a higher local peak
				for j in range(1,round((bite_length_sec*DATA_FREQ/STRIDE)-0.5)):
					if (i+j)>=len(predictions_LM):
						break
					if predictions_LM[i+j]==1:
						if predictions[running_max_index]<predictions[i+j]:
							# erase previous LM since it is "covered up" by new detection placement
							predictions_LM[running_max_index]=0
							# mark new location of running max 
							running_max_index=i+j
						else:
							predictions_LM[i+j]=0 # Cover up the smaller local maxima
				# Add detection, then cover remaining points up to duration seconds ahead
				Detections.append(timesteps[running_max_index]+delayed_offset_sec)
				for j in range(0,round((bite_length_sec*DATA_FREQ/STRIDE)-0.5)):
					if (running_max_index+j)>=len(predictions_LM):
						break
					if predictions_LM[running_max_index+j]==1:
						predictions_LM[running_max_index+j]=0
			# end of if predictions_LM[i]=1
		#end of for i in range(len(predictions_LM)) loop

	elif DETECTION_METHOD_FLAG == 2: # 2nd Attempt: Grab the maximum in each non-floored segment

		### Knobs ###
		SizeOfGapToMergeSegments=5 #number of predictions needing to be a zero to close off a segment

		### Other Variables ###
		FinishedFlag1=0 # used to mark end of loop across all predictions
		rover1=0 # Main iterator marking where in the prediciton file has been fully inspected
		rover2=0 # when main rover finds a segment, rover 2 scouts ahead to find the 
				# first datapoint after the segment (or end of list)
		Detections=[] # initialize as empty list, add detections as they are found

		while (FinishedFlag1!=1):
			if predictions[rover1]==0: #if no detection, move to next bite
				rover1+=1
				if rover1>=len(predictions)-1: #add the "-1" in order to leave space for rover2 to be inserted
					FinishedFlag1=1
			else: #if the prediction is the start of a bite segment
				FinishedFlag2=0
				rover2=rover1+1 #start scout just ahead of current detected start
				while FinishedFlag2!=1:
					if predictions[rover2]==0:
						FinishedFlag2=1 # Begin with assumption one will end loop if this is the end of segment
						# start at one and add up to SizeOfGapToMergeSegments
						for i in range(1,SizeOfGapToMergeSegments+1):
							# set rover to end if the end of the data is reached
							if rover2+i>=len(predictions):
								rover2=len(predictions)-1
								break # leave i loop and end while since Flag is still false
							if predictions[rover2+i]!=0: #if the gap is short and a non-zero bite is found
								rover2=rover2+i # move scout to the non-zero point
								FinishedFlag2=0 # correct assumption since the gap was short
								break # and continue search in while() loop
						# if loop is exited and FinishedFlag2 has not been cleared, then rover2 is at end
					else: # if predictions[rover2]!=0
						rover2+=1 # move rover along if segment is still occuring
						if rover2>=len(predictions): # check boundary for end of data
							rover2=len(predictions)-1
							FinishedFlag2=1 # or use break statement
				# end of while FinishedFlag2!=1


				# Add Detection as the maximum of the segment
				currMax=predictions[rover1] #start with first point as max
				currMaxIndex=rover1
				for rover3 in range(rover1+1,rover2): # iterate over all remaining data in segment
					if predictions[rover3]>=currMax: # biased to later times if a tie
						currMax=predictions[rover3] 
						currMaxIndex=rover3
				Detections.append(timesteps[currMaxIndex]+delayed_offset_sec)

				# Shift Rover to end of Segment and continue
				rover1=rover2
				if rover1>=len(predictions)-1: #set to minus one from end so that rover2=rover1+1 won't exceed index
					FinishedFlag1=1 #could also just use a break statement

		# end of while FinishedFlag==0
	# end of predictions[] --> Detections[] if statement







	##################################
	####  MATCH DETECTIONS TO GT  ####
	##################################



	# Convert to np-array to use Eval function 
	Detections = np.array(Detections)
	gt_bite_times = np.array(currGT_times)

	if (len(Detections)==0):
		print("Zero Detections for meal index {}".format(curr_meal))
		print("len = {}, shape = ".format(len(gt_bite_times)),end='')
		print(np.shape(gt_bite_times))
		TP = 0
		FP = 0
		FN = len(gt_bite_times)

	elif EVAL_METHOD_FLAG == 1:
		[TP, FP, FN, Key] = DongEval_PtVsPt(
			Detections, gt_bite_times
		)
	elif EVAL_METHOD_FLAG == 2:
		# GT Should have already be converted to Windows
		[TP, FP, FN, FP1, FP2, Key] = KyritEval_PtVsWindow(
			Detections, gt_bite_times,WIN_TOLERANCE = WIN_TOLERANCE
		)
		All_FP1.append(FP1)
		All_FP2.append(FP2)
	elif EVAL_METHOD_FLAG == 3:
		# Convert Predictions to windows
		Detections_wind = TimePoint2Window(Detections)
		# GT Should have already be converted to Windows
		[TP, FP, FN, TN, Key] = Duration_WindowVsWindow(
			Detections_wind, gt_bite_times
		)
	else:
		print("ERROR: UNKNOWN EVAL METHOD REQUESTED. Please pass in 1 for Dong_PtVSPt, 2 for Kyritsis_PtVsWindow, and 3 for Duration_WindVsWind.")
		raise Exception("ERROR: UNKNOWN EVAL METHOD REQUESTED. Please pass in 1 for Dong_PtVSPt, 2 for Kyritsis_PtVsWindow, and 3 for Duration_WindVsWind.")
	#end of EVAL_METHOD_FLAG Switch



	All_TP.append(TP)
	All_FP.append(FP)
	All_FN.append(FN)


	if DEBUG_PRINTS == 1:
		print("Through meal {} of {}...".format(curr_meal, len(compiled_eval_data)))
	#end of DEBUG_PRINTS
# end of for curr meal loop

print("Finished Evaluating Model."


TP = 102, FP = 4, FN = 868
TP=102 FP=4 U=868
	FP1=0 FP2=4
TPR=10.515
PPV=96.226
F1_Score=18.959


In [5]:
TP = sum(All_TP) # overwrite the short names with the total results
FP = sum(All_FP)
FN = sum(All_FN)

print("TP = {}, FP = {}, FN = {}".format(TP,FP,FN))
if EVAL_METHOD_FLAG == 2:
	FP1 = sum(All_FP1)
	FP2 = sum(All_FP2)
	#Verbose flag set to 0
	PrintStats(TP, FP, FN, FP1 = FP1, FP2 = FP2,
			  VerbosePrintOpt = 0
			  )
else:
	PrintStats(TP, FP, FN)

if DEBUG_PRINTS == 1:
	print("FINISHED EVALUATING")
#end DEBUG_PRINTS

#PlotFIC_GT_Preds(currGT_times,gt_matched,predictions,timesteps,Detections,DetectionOutcomes)


TP = 102, FP = 4, FN = 868
TP=102 FP=4 U=868
	FP1=0 FP2=4
TPR=10.515
PPV=96.226
F1_Score=18.959


In [ ]:
model
